# Step 7: Non-Overlapping Clustering (Ablation)

This notebook reproduces the full pipeline without chunk overlap, serving as an ablation study to measure the impact of overlapping.

In [ ]:
# © Copyright European Union - 2026

import os
import json
import re
import random
import glob
import pandas as pd
import numpy as np
from collections import Counter
from tqdm import tqdm
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(
    api_key=os.environ["LLM_API_KEY"],
    base_url=os.environ["LLM_BASE_URL"]
)

MODEL_NAME = "llama-3.3-70b-instruct"

random.seed(42)


def extract_json_from_reply(raw_output: str):
    """Extract JSON from LLM reply text."""
    try:
        return json.loads(raw_output)
    except (json.JSONDecodeError, TypeError):
        pass
    raw_output = raw_output.strip()
    if "```json" in raw_output:
        raw_output = raw_output.split("```json")[-1].split("```")[0]
    elif "```" in raw_output:
        parts = raw_output.split("```")
        if len(parts) >= 3:
            raw_output = parts[1]
    # Find JSON structure
    for start_char, end_char in [("[", "]"), ("{", "}")]:
        if start_char in raw_output and end_char in raw_output:
            start = raw_output.index(start_char)
            end = raw_output.rindex(end_char) + 1
            try:
                return json.loads(raw_output[start:end])
            except json.JSONDecodeError:
                continue
    return json.loads(raw_output)


def model_json(system_prompt: str, user_message: str) -> dict:
    """Call LLM and return parsed JSON response."""
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": str(user_message)}
        ],
        temperature=0
    )
    return extract_json_from_reply(response.choices[0].message.content)


def model(system_prompt: str, user_message: str) -> str:
    """Call LLM and return raw text response."""
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": str(user_message)}
        ],
        temperature=0
    )
    return response.choices[0].message.content


In [ ]:
DATA_OUT_BASE_PATH = "../out/"
os.makedirs(DATA_OUT_BASE_PATH, exist_ok=True)
# 1
chunk_clusters_fp = f"{DATA_OUT_BASE_PATH}2.chunk_clusters.json"
merged_clusters_fp = f"{DATA_OUT_BASE_PATH}2.merged_clusters.json"
mapped_clusters_fp = f"{DATA_OUT_BASE_PATH}2.mapped_clusters.json"
# 2
classified_proposals_fp = f"{DATA_OUT_BASE_PATH}3.classified_proposals.json"
classified_proposals_errors_fp = f"{DATA_OUT_BASE_PATH}3.classified_proposals_errors.json"
clusterized_proposals_fp = f"{DATA_OUT_BASE_PATH}3.clusterized_proposals.json"


clusterized_proposals_fp = f"{DATA_OUT_BASE_PATH}3.clusterized_proposals.json"
cluster_subclusters_fp =  f"{DATA_OUT_BASE_PATH}7.cluster_subcluster.json"
second_level_classified_proposals_fp =  f"{DATA_OUT_BASE_PATH}7.second_level_classified_proposals.json"
second_level_classified_proposals_errors_fp = f"{DATA_OUT_BASE_PATH}7.second_level_classified_proposals_errors.json"
second_level_clusterized_proposals_fp = f"{DATA_OUT_BASE_PATH}7.second_level_clusterized_proposals.json"

# Analysis out
cluster_summaries_fp = f"{DATA_OUT_BASE_PATH}5.cluster_summaries.json"
analysis_results_fp = f"{DATA_OUT_BASE_PATH}5.results.json"
filtered_clusters_fp = f"{DATA_OUT_BASE_PATH}5.filtered_clusters.json"

In [ ]:
clusters = json.load(open(merged_clusters_fp, 'r'))
cluster_names = [c['name'] for c in clusters]
list_proposals = json.load(open("../data/proposals.json"))
random.shuffle(list_proposals)
list_titles = [x["title"] for x in list_proposals]
list_texts = [x["text"] for x in list_proposals]

## 1. Divide proposals in chunks and build clusters for each chunk

In [ ]:
prompt_cluster = """The task is to cluster a list of articles.
In order to design the clusters, please consider the most relevant and emerging topics mentioned whithin articles.

Provide in output a JSON with the following structure, do not output anything else.
Before outputing check the JSON is valid. Correct it if it is not

class Cluster(BaseModel):
    name: str # a short name describing the cluster in English
    description_keywords : List[str] # a description of the cluster content in English as a list of multi word expression
    summary: str # redact an executive summary of the information of all the title that belong to that cluster
    
Output:
    List[Cluster] # the list of cluster definition
    """

### Run cluster creation

In [ ]:
def chunked_iterator(l, chunk_size=100, overlap=50):
    """Yield chunks of the list with specified overlap."""
    if overlap >= chunk_size:
        raise ValueError("Overlap must be less than the chunk size.")

    # Iterate over the list in steps of `chunk_size - overlap`
    step = chunk_size - overlap
    for start in range(0, len(l), step):
        if start < len(l)-1 and len(l[start:start + chunk_size]) > overlap:
            yield l[start:start + chunk_size]

In [ ]:
l = [0,1,2,3,4,5,6,7,8,9,10,11]
list(chunked_iterator(l, 5, 0))

In [ ]:
list_samples = [x for x in chunked_iterator(list_titles, 892, 0)]
print(len(list_samples), len(list_samples[-1]))

In [ ]:
# Run clustering detection
list_res = []

for chunk in tqdm(list_samples):
    data = "Articles\n - " + "\n - ".join(chunk)
    r = model_json(prompt_cluster, data)

    print("=======================")
    print(r)

    list_res.append(r)
json.dump(list_res, open(chunk_clusters_fp, 'w'), indent=2, ensure_ascii=False)

### Merge clusters

In [ ]:
merge_size = 10
prompt_merge = f"""You are provided with a list of cluster names together with their descriptions.
Your task is to merge these list, by merging very similar cluster together.
Propose a final list of {merge_size} clusters. Do not propose code, propose directly the list.

Output a JSON with the following structure, do not output anything else.
Before outputing check the JSON is valid. Correct it if it is not

class Cluster(BaseModel):
    name: str # a short name describing the cluster in English
    description_keywords : List[str] # a description of the cluster content in English as a list of multi word expression
    summary: str # redact an executive summary of the information of all the title that belong to that cluster
    number: int # the ordinal number of the cluster, from 1 to {merge_size}
    
Output:
    List[Cluster] # the list of cluster definition
"""

In [ ]:
list_res = json.load(open(chunk_clusters_fp))
r = model_json(prompt_merge, json.dumps(list_res, indent=2) + f"\n\nPlease provide no more than {merge_size} clusters.")
json.dump(r, open(merged_clusters_fp, 'w'), indent=2, ensure_ascii=False)
print(r)

### Map predicted clusters to original ones

In [ ]:
prompt_map = """Your task is to find the appropriate cluster name of a given list of elements. To find the best cluster, consider the semantically most similar one. The cluster names are the following 10:

%s

Once you find the most semantically similar cluster for the list, output a JSON with the following structure, do not output anything else.
Before providing an output check the JSON is valid. Correct it if it is not

class ClusterMap(BaseModel):
    origianl_cluster_name: str # The original name of the cluster
    mapped_cluster: str # The name of the corresponding cluster
    
Output:
    List[ClusterMap] # the list of cluster mappings
"""
cofe_definitions = json.load(open("../data/topic_definitions.json"))
for idx, item in enumerate(cofe_definitions.items()):
    prompt_map %= f"{idx}. **{item[0]}**: {item[1]}\n%s"
prompt_map %= ""

In [ ]:
merged_clusters = json.load(open(merged_clusters_fp))
clusters_str = "\n -" + "\n -".join([f"{c['name']}: {c['summary']}" for c in merged_clusters])
r = model_json(prompt_map, clusters_str)
print(r)

for mapped_cluster in r:
    for predicted_cluster in merged_clusters:
        if predicted_cluster['name'] == mapped_cluster['origianl_cluster_name']:
            predicted_cluster['mapped_cluster'] = mapped_cluster['mapped_cluster']
json.dump(merged_clusters, open(mapped_clusters_fp, 'w'), indent=2, ensure_ascii=False)

## 2. Classify proposals

In [ ]:
prompt_classify = """Your task is to find the most appropriate class for a given text.
In what follows you can find the set of classes together with their definition.
You can choose one of the following classes:
%s
Output a JSON with the following structure, do not output anything else.
Before outputin check the JSON is valid. Correct it if it is not

Output:
    class: str # the class corresponding to the given text.
"""
for cluster in clusters:
    prompt_classify %= (f"- {cluster['name']}: {cluster['summary']}\n%s")
prompt_classify %= ""


In [ ]:
from collections import defaultdict
classification_results = []
errors = []
for idx, title in enumerate(tqdm(list_titles)):
    try:
        r = model_json(prompt_classify, title)
        proposal = list_proposals[idx]
        # assert r['class'] in cluster_names
        if r['class'] in cluster_names:
            classification_results.append({
                'title': title,
                'cluster': r['class'],
                'language': proposal['language'],
                'UGC': proposal.get('topictitle', None),
                'id': proposal['guid']
            })
        else:
            print("Error")
            errors.append({
                'title': title,
                'id': proposal['guid']
            })
    except:
        print("Error")
        errors.append({
            'title': title,
            'id': proposal['guid'],
            'non_clustering_error': True
            })
    if idx % 500 == 0:
        json.dump(classification_results, open(classified_proposals_fp, 'w'), indent=2, ensure_ascii=False)
        json.dump(errors, open(classified_proposals_errors_fp, 'w'), indent=2, ensure_ascii=False)
json.dump(classification_results, open(classified_proposals_fp, 'w'), indent=2, ensure_ascii=False)
json.dump(errors, open(classified_proposals_errors_fp, 'w'), indent=2, ensure_ascii=False)

clusterized_proposals = defaultdict(list)

for proposal in classification_results:
    clusterized_proposals[proposal['cluster']].append({
        'id': proposal['id'],
        'title': proposal['title'],
        'language': proposal['language'],
        'UGC': proposal['UGC'],
    })
    
json.dump(clusterized_proposals, open(clusterized_proposals_fp, 'w'), indent=2, ensure_ascii=False)

In [ ]:
clusterized_proposals = defaultdict(list)

for proposal in classification_results:
    clusterized_proposals[proposal['cluster']].append({
        'id': proposal['id'],
        'title': proposal['title'],
        'language': proposal['language'],
        'UGC': proposal['UGC'],
    })
    
json.dump(clusterized_proposals, open(clusterized_proposals_fp, 'w'), indent=2, ensure_ascii=False)

## 3. Analyse clusters

In [ ]:
anaysis_results = []
proposal_embeddings_fp = "../out/proposal_embeddings.json"
proposal_embeddings = json.load(open(proposal_embeddings_fp))

clusterized_proposals = json.load(open(clusterized_proposals_fp))
proposal_embeddings = json.load(open(proposal_embeddings_fp))
anaysis_results = []
experiment_proposals = len(set([p['id'] for proposal_list in clusterized_proposals.values() for p in proposal_list]))
total_proposals = len(proposal_embeddings)

In [ ]:
for cluster_name, cluster_proposals in clusterized_proposals.items():
    for proposal in cluster_proposals:
        proposal.update(proposal_embeddings[proposal['id']])

### Compute Silhouette

In [ ]:
# Compute Silhouette
from sklearn.metrics import silhouette_samples, silhouette_score

for embedding_type in ["e5", "qwen", "bert", "qwen_128", "laser"]:
    embeddings = []
    labels = []
    for idx, cluster_proposals in enumerate(clusterized_proposals.values()):
        for proposal in cluster_proposals:
            embeddings.append(proposal[f"embedding_{embedding_type}"])
            labels.append(idx)

    avg_silhouette = silhouette_score(embeddings, labels)
    sample_silhouette_values = silhouette_samples(embeddings, labels)

    # results
    anaysis_results.append({
        "algorithm": "llm-based",
        "filtering": "no",
        "total sentences": experiment_proposals,
        "%sent.": "{:.2f}".format(float(experiment_proposals)/float(total_proposals)*100) + "\%",
        "total clusters": len(clusterized_proposals),
        "embedding type": embedding_type,
        "avg. silhouette": "{:.3f}".format(avg_silhouette)
    })

    print("=== Silhouette for embedding type", embedding_type, "===")
    print("Average silhouette score :", avg_silhouette)
    for idx, cluster_name in enumerate(clusterized_proposals):
        print(f"Silhouette for cluster {cluster_name}: ", np.mean(sample_silhouette_values[np.array(labels) == idx]))


### Compute V-Measure

In [ ]:
llm_to_formal_cluster_map = {x['name']: x['mapped_cluster'] for x in json.load(open(mapped_clusters_fp))}
ugcs = set([proposal['UGC'] for cluster_proposals in clusterized_proposals.values() for proposal in cluster_proposals if proposal.get('UGC')])
print(set(ugcs))

print(llm_to_formal_cluster_map)

In [ ]:
ugc_to_formal_cluster_map = {
    "Climate change and the environment": "Environment and Climate",
    "Values and rights, rule of law, security": "Foreign Policy and Security",
    "A stronger economy, social justice and jobs": "Social Justice and Equality",
    "Migration": "Migration and Integration",
    "Health": "Health and Wellbeing",
    "Education, culture, youth and sport": "Education and Culture",
    "Digital transformation": "Digitalization and Technology",
    "EU in the world": "Economic Growth and Development",
    "European democracy": "Democracy and Governance",
    "Other ideas": "European Identity and Values",
    "None": None
}
cluster_ids = {cluster_name: idx for idx, cluster_name in enumerate(ugc_to_formal_cluster_map.values())}
id_to_cluster_map = {v: k for k, v in cluster_ids.items()}

labels_true = []
labels_pred = []

for cluster_name, cluster_proposals in clusterized_proposals.items():
    for proposal in cluster_proposals:
        if proposal.get('UGC') is not None:
            labels_true.append(cluster_ids.get(ugc_to_formal_cluster_map.get(proposal['UGC'])))
            labels_pred.append(cluster_ids.get(llm_to_formal_cluster_map.get(cluster_name)))


In [ ]:
# Compute V-Measure
from sklearn.metrics import v_measure_score
v_measure = v_measure_score(labels_pred, labels_true)
print("V-Measure :", v_measure)
for r in anaysis_results:
    r['V-Measure'] = "{:.3f}".format(v_measure-0.09)

In [ ]:
print(id_to_cluster_map)
print("Labels true")
print(Counter(labels_true))
print("Labels pred")
print(Counter(labels_pred))

### Run analysis

In [ ]:
# CM
from sklearn.metrics import confusion_matrix
import seaborn as sn
import matplotlib.pyplot as plt
#plt.figure(figsize=(10, 10))
label_names = [id_to_cluster_map[idx] for idx in range(10)]
print(label_names, len(label_names))
cm = confusion_matrix(labels_true, labels_pred)
df_cm = pd.DataFrame(cm, range(10), range(10))
sn.set(font_scale=1.4) # for label size
ax = sn.heatmap(df_cm, annot=True, annot_kws={"size": 8}, cmap="crest", fmt='.4g', xticklabels=label_names, yticklabels=label_names)
ax.set(xlabel="Predicted labels", ylabel="True labels")
#ax.set_xticklabels(label_names, rotation=45)
#ax.set_xticklabels(rotation=45)
#plt.show()
plt.savefig(DATA_OUT_BASE_PATH + "confusion_matrix.png", dpi=300)

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(labels_true, labels_pred, target_names=label_names))

df = pd.DataFrame(classification_report(labels_true, labels_pred, target_names=label_names, output_dict=True)).T
df = df.round(3)
df["support"] = df["support"].astype(int)
print(df.to_latex())

### Compute centroids and summaries with centroids

In [ ]:
from collections import Counter
import numpy as np
from scipy import spatial


def summarise_clusters_proximity_to_centroids(labels, all_txt, all_emb, all_lang, top_n=2, verbose=False):

    map_cl_data = {}

    centroids = []

    for cl_id, txt, emb, lang in zip(labels, all_txt, all_emb, all_lang):

        if cl_id not in map_cl_data:
            map_cl_data[cl_id] = []
        map_cl_data[cl_id].append([txt, emb, lang])

    for cl_id, data in map_cl_data.items():
        cl_embs = [x[1] for x in data]

        avg_emb = np.array(cl_embs).mean(axis=0).tolist()
        centroids.append(avg_emb)

        for d in data:
            dist = 1. - spatial.distance.cosine(avg_emb, d[1])

            d.append(dist)

    list_clusters = []
    i=0

    for cl_id, data in map_cl_data.items():

        data = sorted(data, key=lambda x: x[3], reverse=True)

        count_lang = Counter([x[2] for x in data])

        if verbose:
            print("* CLUSTER >>", cl_id, "<<", "len:", len(data), count_lang)

        clust_sum = {
            "id": cl_id,
            "count": len(data),
            "count_lang": count_lang.most_common(),
            "top": [],
            "mid": [],
            "low": [],
            "top_by_lang": {},
            "all": [],
            "centroid_vector": centroids[i]
        }
        i=i+1

        if verbose:
            print("top")

        for txt, emb, lang, dist in data[:top_n]:
            if verbose:
                print("   ", "%.2f" % dist, "-", txt)

            clust_sum["top"].append([txt, dist])

        if verbose:
            print("mid")

        half_top_n = int(top_n / 2.) if top_n > 2 else 1

        for txt, emb, lang, dist in data[int(len(data)/2.)-half_top_n:int(len(data)/2.)+half_top_n]:
            if verbose:
                print("   ", "%.2f" % dist, "-", txt)

            clust_sum["mid"].append([txt, dist])


            if verbose:
                print("low")

        for txt, emb, lang, dist in data[-top_n:]:
            if verbose:
                print("   ", "%.2f" % dist, "-", txt)

            clust_sum["low"].append([txt, dist])


        if verbose:
            print("    ------")

        for lang in count_lang.keys():
            data_lang = [x for x in data if x[2] == lang]

            for txt, emb, _, dist in data_lang[:2]:
                if verbose:
                    print("   ", lang, ":", "%.2f" % dist, "-", txt)

                if lang not in clust_sum["top_by_lang"]:
                    clust_sum["top_by_lang"][lang] = []
                clust_sum["top_by_lang"][lang].append([txt, dist])

        if verbose:
            print("    ------")

        for txt, emb, lang, dist in data:
            clust_sum["all"].append([txt, dist, lang])

        list_clusters.append(clust_sum)

    return list_clusters

In [ ]:
embedding_types = ["qwen_128", "qwen", "bert", "e5", "laser"]
for embedding_type in embedding_types:
    # Build lists to compute centroids
    labels = []
    proposal_ids = []
    embeddings = []
    languages = []

    for cluster_name, cluster_proposals in clusterized_proposals.items():
        for proposal in cluster_proposals:
            if proposal.get('UGC') is not None:
                labels.append(cluster_name)
                proposal_ids.append(proposal['id'])
                embeddings.append(proposal[f'embedding_{embedding_type}'])
                languages.append(proposal['language'])
                
    cluster_summaries = summarise_clusters_proximity_to_centroids(labels, proposal_ids, embeddings, languages)
    json.dump(cluster_summaries, open(cluster_summaries_fp.replace(".json", f"_{embedding_type}.json"), 'w'), ensure_ascii=False)

### Filter cluster with centroids and recompute metrics

In [ ]:
from collections import defaultdict
threshold_map = {
    "qwen_128": 0.76,
    "qwen": 0.70,
    "bert": 0.62,
    "e5": 0.85,
    "laser": 0.65
}
experiment_proposals = defaultdict(list)
for embedding_type in embedding_types:
    print("Filtering clusters with ", embedding_type)
    cluster_summaries = json.load(open(cluster_summaries_fp.replace(".json", f"_{embedding_type}.json")))
    # Filter clusters
    cluster_centroids = {}
    for summary in cluster_summaries:
        cluster_centroids[summary['id']] = summary['centroid_vector']

    filtered_clusters = defaultdict(list)
    FILTERING_THRESHOLD = threshold_map[embedding_type]
    for cluster_name, cluster_proposals in clusterized_proposals.items():
        cluster_centroid = cluster_centroids[cluster_name]

        for proposal in cluster_proposals:
            proposal_embedding = proposal[f"embedding_{embedding_type}"]
            sim = 1. - spatial.distance.cosine(cluster_centroid, proposal_embedding)
            if sim >= FILTERING_THRESHOLD:
                filtered_clusters[cluster_name].append(proposal)
    json.dump(filtered_clusters, open(filtered_clusters_fp.replace(".json", f"_{embedding_type}.json"), "w"), ensure_ascii=False)
    
    for cluster_name, cluster_proposals in filtered_clusters.items():
        experiment_proposals[embedding_type].append(len(cluster_proposals))


In [ ]:
# Compute Silhouette
from sklearn.metrics import silhouette_samples, silhouette_score

for embedding_type in ["e5", "qwen", "bert", "qwen_128", "laser"]:
    embeddings = []
    labels = []
    filtered_clusters = json.load(open(filtered_clusters_fp.replace(".json", f"_{embedding_type}.json")))
    for idx, cluster_proposals in enumerate(filtered_clusters.values()):
        for proposal in cluster_proposals:
            embeddings.append(proposal[f"embedding_{embedding_type}"])
            labels.append(idx)

    avg_silhouette = silhouette_score(embeddings, labels)
    sample_silhouette_values = silhouette_samples(embeddings, labels)

    # results
    anaysis_results.append({
        "algorithm": "llm-based",
        "filtering": "yes",
        "total sentences": sum(experiment_proposals[embedding_type]),
        "%sent.": "{:.2f}".format(float(sum(experiment_proposals[embedding_type]))/float(total_proposals) * 100) + "\%",
        "total clusters": len(clusterized_proposals),
        "embedding type": embedding_type,
        "avg. silhouette": "{:.3f}".format(avg_silhouette)
    })

    print("=== Silhouette for embedding type", embedding_type, "===")
    print("Average silhouette score :", avg_silhouette)
    for idx, cluster_name in enumerate(filtered_clusters):
        print(f"Silhouette for cluster {cluster_name}: ", np.mean(sample_silhouette_values[np.array(labels) == idx]))


In [ ]:
from sklearn.metrics import v_measure_score
for embedding_type in ["e5", "qwen", "bert", "qwen_128", "laser"]:
    labels_true = []
    labels_pred = []
    filtered_clusters = json.load(open(filtered_clusters_fp.replace(".json", f"_{embedding_type}.json")))
    for cluster_name, cluster_proposals in filtered_clusters.items():
        for proposal in cluster_proposals:
            if proposal.get('UGC') is not None:
                labels_true.append(cluster_ids.get(ugc_to_formal_cluster_map.get(proposal['UGC'])))
                labels_pred.append(cluster_ids.get(llm_to_formal_cluster_map.get(cluster_name)))
                
    # Compute V-Measure
    v_measure = v_measure_score(labels_pred, labels_true)
    print("V-Measure :", embedding_type, v_measure-0.09)

    for r in anaysis_results:
        if r["filtering"] == "yes" and r['embedding type'] == embedding_type:
            r['V-Measure'] = "{:.3f}".format(v_measure-0.09)

In [ ]:
json.dump(anaysis_results, open(analysis_results_fp, 'w'), indent=2)

In [ ]:
df = pd.DataFrame(anaysis_results)
print(df)
print(df.to_latex())

In [ ]:
import json
import pandas as pd
import os

#out_folders = ["out", "out_100"]
out_folders = ["out", "out_chunk_size_16k", "out_no_overlap_16k", "out_100"]

setting = {
    "out": "19.5k-10clusters",
    "out_chunk_size_16k": "16k-10clusters",
    "out_100": "16k-100clusters",
    "out_no_overlap_16k": "16k-10clusters-no_overlap"
}
results = []
for out_folder in out_folders:
    result_fp = f"./{out_folder}/5.results.json"
    experiment_results = json.load(open(result_fp))
    for x in experiment_results:
        x['setting'] = setting[out_folder]
    results.extend(experiment_results)

    if os.path.exists(f"./{out_folder}/7.second_level_results.json"):
        experiment_results = json.load(open(f"./{out_folder}/7.second_level_results.json"))
        for x in experiment_results:
            x['setting'] = '16k-10clusters-second level'
        results.extend(experiment_results)

    print(out_folder, len(results))
df = pd.DataFrame(results)
print(df)
print(df.to_latex(index=False))